<img src='./img/up_bann.png' style="display: block; margin: 0 auto;">

*This algorithm has been **community** contributed to APEx and has not undergine scientific validation review* 

# Kakhovka dam breach — flood detection, Kherson (June 2023)

The dam on the Dnipro was destroyed on 6 June 2023 and flooded the river downstream. The notebook runs the `flood_detection` algorithm twice, once before the breach and once during the flood. It marks water using **Sentinel-1** radar, which sees through cloud, and lays it over a **Sentinel-2** background.

Scenes picked up by each window (openEO end dates are exclusive):

| Run | Window | Sentinel-1 GRD | Sentinel-2 L2A |
|---|---|---|---|
| Pre-flood | 2023-06-02 → 2023-06-04 | 2023-06-02 (ascending, full bbox) | 2023-06-03 (8% cloud) |
| During flood | 2023-06-03 → 2023-06-10 | 2023-06-09 (ascending, east ~68%) + 2023-06-06 (west edge) | 2023-06-03 |

### Import Required libraries

In [ ]:
from pathlib import Path
import os
from getpass import getpass
from esa_apex_toolbox.algorithms import GithubAlgorithmRepository

import openeo

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.patches import Patch

### Define Costum Functions

In [ ]:
# Read at 1/3 resolution: a full 10 m scene is ~6800x5000 px per panel.
DOWNSAMPLE = 3

def read_rgb(path):
    """Read a 3-band flood_detection GeoTIFF as an (H, W, 3) 0-1 array plus its extent in km."""
    with rasterio.open(path) as src:
        out_shape = (3, src.height // DOWNSAMPLE, src.width // DOWNSAMPLE)
        data = src.read(out_shape=out_shape)
        b = src.bounds
    # The UDP output is already display-scaled; only clip to 0-1.
    rgb = np.clip(np.nan_to_num(np.moveaxis(data, 0, -1)), 0, 1)
    extent = [b.left / 1000, b.right / 1000, b.bottom / 1000, b.top / 1000]
    return rgb, extent

### Authentication

To run the jobs directly in the openEO CDSE federation, ir is required to authenticate with our CDSE credentials:

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu/").authenticate_oidc()

Also, to explore the APEx Catalog, a GitHub token is recommended (GitHub normally allows 60 unauthenticated **REST API requests per hour per IP address**):

In [ ]:
github_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
if not github_token:
    github_token = getpass('GitHub token (optional; Enter for unauthenticated access): ') or None

We define our area of interest, the time frame of interest and our output paths

In [ ]:
# Bounding box and temporal extent

shape = {"west": 31.900898, "south": 46.333770, "east": 32.777495, "north": 46.773495}
pre_flood_start_date, pre_flood_end_date = ["2023-06-02", "2023-06-04"]
flood_start_date, flood_end_date = ["2023-06-03", "2023-06-10"]

pre_flood_output = Path("flood_detection_kherson_preflood.tif")
flood_output = Path("flood_detection_kherson_flood.tif")


Next step is selecting our algorithm and accessing the `udp_link` trough its `id`. For this case it corresponds to `flood_detection`

In [ ]:
repo = GithubAlgorithmRepository(
    owner='ESA-APEx',
    repo='apex_algorithms',
    folder='algorithm_catalog',
    branch='main',
    token=github_token,
)

process_id = "flood_detection"
fd_algorithm = repo.get_algorithm(process_id)

fd_algorithm

As we are interested in **pre-flood** and **during flood** events, we define two data cubes ; this means two separate jobs.

In [ ]:
pre_flood_kherson = connection.datacube_from_process(
    process_id,
    namespace=fd_algorithm,
    time=[pre_flood_start_date, pre_flood_end_date],
    bounding_box=shape,
)

flood_kherson = connection.datacube_from_process(
    process_id,
    namespace=fd_algorithm,
    time=[flood_start_date, flood_end_date],
    bounding_box=shape,
)

## Pre-flood

We prepare and monitor our **pre-flood** job

In [ ]:
pre_flood_job = pre_flood_kherson.create_job(
    title=f"Pre-flood detection for {pre_flood_start_date} to {pre_flood_end_date}",
    out_format="GTiff",
)
pre_flood_job.start_and_wait()

And save our results for visualisation:

In [ ]:
# Each result is a single GeoTIFF (~360 MB). If the download breaks, rerun only
# this cell; the job does not need to be resubmitted.
pre_flood_job.get_results().download_file(pre_flood_output)

## During flood

Then, we can start our **during flood** job, run and monitor.

In [ ]:
flood_job = flood_kherson.create_job(
    title=f"Flood detection for {flood_start_date} to {flood_end_date}",
    out_format="GTiff",
)
flood_job.start_and_wait()

And, retrieve as well our results for further processing in any GIS or software of interest.

In [ ]:
flood_job.get_results().download_file(flood_output)

## Pre-flood vs during flood

Based on our defined `read_rgb` funciton, we can explore our generated results and compare the impact the breach caused.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), squeeze=False)

for ax, path, title in (
    (axes[0, 0], pre_flood_output, "Pre-flood — S1 2023-06-02, S2 2023-06-03"),
    (axes[0, 1], flood_output, "During flood — S1 2023-06-09 (west edge 06-06), S2 2023-06-03"),
):
    rgb, extent = read_rgb(path)
    ax.imshow(rgb, extent=extent)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Easting (km, UTM 36N)")
    ax.set_ylabel("Northing (km, UTM 36N)")

fig.legend(
    handles=[
        Patch(color="#2a2adc", label="Water (Sentinel-1 VV ≤ −15 dB)"),
        Patch(color="#9a9a9a", label="Land (Sentinel-2 NIR background)"),
    ],
    loc="lower center",
    ncol=2,
    frameon=False,
    fontsize=11,
)
fig.suptitle("Kakhovka dam breach — lower Dnipro flooding, Kherson (June 2023)", fontsize=16)
fig.tight_layout(rect=(0, 0.1, 1, 0.95))
fig.savefig("flood_detection_kherson_comparison.png", dpi=200)
plt.show()

The figure shows two maps side by side, before and during the flood. <br>

Water is blue and land is grey. In the "during" panel, the islands and floodplain of the Dnipro delta are underwater. This shows how radar can map damage to critical infrastructure within days, whatever the weather. Note that the "during" image is made from two Sentinel-1 passes, so you can see a seam near the western edge.

<img src='./img/low_bann.png' style="display: block; margin: 0 auto;">